# 인공지능과 금융공학 - TinyGPT 통합 모델 (금융 뉴스 헤드라인 버전)

이 노트북은 기존의 Shakespeare 문학 텍스트 대신, 주가 예측에 자주 사용되는 **금융 및 글로벌 뉴스 헤드라인 데이터셋(Combined_News_DJIA.csv)**을 다운로드하고 정제하여 TinyGPT를 학습시키는 실습용 주피터 노트북입니다. 
데이터 다운로드, 파싱/전처리, 토큰화, 모델 정의, 학습/검증 루프, 그리고 최종 금융 뉴스 헤드라인 텍스트 예측 과정까지 일체형으로 통합되어 있습니다.

In [ ]:
import os
import csv
import urllib.request
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

## 1. 하이퍼파라미터 설정

In [ ]:
# 하이퍼파라미터 정의
DATA_URL = "https://raw.githubusercontent.com/gakudo-ai/open-datasets/refs/heads/main/Combined_News_DJIA.csv"
DATA_FILE = "Combined_News_DJIA.csv"

BLOCK_SIZE = 64        # 컨텍스트 길이 (문맥 창 크기)
EMB_DIM = 128          # 임베딩 차원수
NUM_HEADS = 4          # 멀티헤드 어텐션의 헤드 개수
NUM_LAYERS = 4         # 트랜스포머 블록 쌓는 개수
DROPOUT = 0.1          # 드롭아웃 확률

BATCH_SIZE = 64
LEARNING_RATE = 3e-4
EPOCHS = 10            # Colab 실습 속도를 위해 10으로 설정
MAX_STEPS_PER_EPOCH = 150 # 에포크당 최대 훈련 스텝 제한

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 2. 데이터 다운로드 및 금융 뉴스 정제(Parsing)

In [ ]:
# 데이터 다운로드
if not os.path.exists(DATA_FILE):
    print("Downloading Combined_News_DJIA.csv...")
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)
    print("Download complete.")
else:
    print("Combined_News_DJIA.csv already exists.")

def load_and_clean_news():
    """CSV 파일에서 Top1~Top25 뉴스 헤드라인을 파싱하고 정제합니다."""
    headlines = []
    with open(DATA_FILE, "r", encoding="utf-8", errors="ignore") as f:
        reader = csv.reader(f)
        header = next(reader)  # 헤더 건너뛰기
        for row in reader:
            if len(row) < 3:
                continue
            for col in row[2:]:
                if not col:
                    continue
                text = col.strip()
                if text.startswith("b'") and text.endswith("'"):
                    text = text[2:-1]
                elif text.startswith('b"') and text.endswith('"'):
                    text = text[2:-1]
                text = text.replace("\\'", "'").replace('\\"', '"')
                text = text.strip()
                if text:
                    headlines.append(text)
    return "\n".join(headlines)

text = load_and_clean_news()
print(f"Total corpus length (characters): {len(text)}")

## 3. 토크나이저 및 데이터셋(Train/Val Split)

In [ ]:
class CharTokenizer:
    def __init__(self, text):
        self.chars = sorted(list(set(text)))
        self.vocab_size = len(self.chars)
        self.stoi = {ch: i for i, ch in enumerate(self.chars)}
        self.itos = {i: ch for ch, i in self.stoi.items()}
        
    def encode(self, string):
        return [self.stoi[ch] for ch in string if ch in self.stoi]
        
    def decode(self, indices):
        return "".join([self.itos[i] for i in indices])

tokenizer = CharTokenizer(text)
print("Vocab size:", tokenizer.vocab_size)

# PyTorch 텐서로 변환 후 9:1 비율로 Train/Val 분할
data_tensor = torch.tensor(tokenizer.encode(text), dtype=torch.long)
n = int(0.9 * len(data_tensor))
train_data = data_tensor[:n]
val_data = data_tensor[n:]

class NextTokenDataset(Dataset):
    def __init__(self, data_tensor, block_size):
        self.data = data_tensor
        self.block_size = block_size
    def __len__(self):
        return len(self.data) - self.block_size
    def __getitem__(self, idx):
        x = self.data[idx : idx + self.block_size]
        y = self.data[idx + 1 : idx + self.block_size + 1]
        return x, y

train_dataset = NextTokenDataset(train_data, BLOCK_SIZE)
val_dataset = NextTokenDataset(val_data, BLOCK_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

## 4. 모델 아키텍처 정의 (TinyGPT)

In [ ]:
class Head(nn.Module):
    def __init__(self, emb_dim, head_size, block_size, dropout=0.1):
        super().__init__()
        self.key = nn.Linear(emb_dim, head_size, bias=False)
        self.query = nn.Linear(emb_dim, head_size, bias=False)
        self.value = nn.Linear(emb_dim, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * (k.size(-1) ** -0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        head_size = emb_dim // num_heads
        self.heads = nn.ModuleList([Head(emb_dim, head_size, block_size, dropout) for _ in range(num_heads)])
        self.proj = nn.Linear(emb_dim, emb_dim)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dropout(out)
        return out

class FeedForward(nn.Module):
    def __init__(self, emb_dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(emb_dim, 4 * emb_dim),
            nn.ReLU(),
            nn.Linear(4 * emb_dim, emb_dim),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, emb_dim, num_heads, block_size, dropout=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(emb_dim)
        self.sa = MultiHeadAttention(emb_dim, num_heads, block_size, dropout)
        self.ln2 = nn.LayerNorm(emb_dim)
        self.ffwd = FeedForward(emb_dim, dropout)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, emb_dim=128, num_heads=4, num_layers=4, dropout=0.1):
        super().__init__()
        self.vocab_size = vocab_size
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, emb_dim)
        self.position_embedding = nn.Embedding(block_size, emb_dim)
        self.blocks = nn.Sequential(*[
            Block(emb_dim, num_heads, block_size, dropout) for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(emb_dim)
        self.lm_head = nn.Linear(emb_dim, vocab_size)
        self.apply(self._init_weights)
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    def forward(self, x):
        B, T = x.shape
        pos = torch.arange(T, device=x.device)
        tok_emb = self.token_embedding(x)
        pos_emb = self.position_embedding(pos)[None]
        h = tok_emb + pos_emb
        h = self.blocks(h)
        h = self.ln_f(h)
        logits = self.lm_head(h)
        return logits

def sequence_cross_entropy(logits, targets):
    return F.cross_entropy(logits.transpose(1, 2), targets)

## 5. 평가(Evaluate) 및 텍스트 생성(Generate) 유틸리티 함수 정의

In [ ]:
@torch.no_grad()
def evaluate(model, loader, device, max_steps=None):
    model.eval()
    total_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(loader):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        total_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if max_steps is not None and step + 1 >= max_steps:
            break
    return total_loss / total_count

@torch.no_grad()
def generate(model, tokenizer, prompt="US ", max_new_tokens=250, block_size=64, device="cpu"):
    model.eval()
    context = torch.zeros((1, block_size), dtype=torch.long, device=device)
    for ch in prompt:
        val = tokenizer.stoi.get(ch, 0)
        context = torch.cat([context[:, 1:], torch.tensor([[val]], device=device)], dim=1)
    out = list(prompt)
    for _ in range(max_new_tokens):
        logits = model(context)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        ix = torch.multinomial(probs, num_samples=1)
        out.append(tokenizer.itos[ix.item()])
        context = torch.cat([context[:, 1:], ix], dim=1)
    return "".join(out)

## 6. 학습 전 무작위 예측 결과 확인

In [ ]:
model = TinyGPT(tokenizer.vocab_size, BLOCK_SIZE, EMB_DIM, NUM_HEADS, NUM_LAYERS, DROPOUT).to(DEVICE)
prompts = ["US ", "Federal Reserve ", "Stock Market "]

print("=== 학습 전 무작위 시드 예측 ===")
for p in prompts:
    print(f"[Prompt: '{p}']")
    print(generate(model, tokenizer, prompt=p, max_new_tokens=150, block_size=BLOCK_SIZE, device=DEVICE))
    print()

## 7. 모델 학습 진행

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
train_losses, val_losses = [], []

start_time = time.time()
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_train_loss, total_count = 0.0, 0
    for step, (xb, yb) in enumerate(train_loader):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        loss = sequence_cross_entropy(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item() * xb.size(0)
        total_count += xb.size(0)
        if step + 1 >= MAX_STEPS_PER_EPOCH:
            break
            
    epoch_train_loss = total_train_loss / total_count
    epoch_val_loss = evaluate(model, val_loader, DEVICE, max_steps=50)
    train_losses.append(epoch_train_loss)
    val_losses.append(epoch_val_loss)
    print(f"Epoch {epoch:2d}/{EPOCHS} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

print(f"Training finished in {time.time() - start_time:.2f} seconds.")

## 8. 학습 완료 후 예측 결과 확인 & Loss Curve 출력

In [ ]:
print("=== 학습 후 예측 뉴스 헤드라인 ===")
for p in prompts:
    print(f"[Prompt: '{p}']")
    print(generate(model, tokenizer, prompt=p, max_new_tokens=250, block_size=BLOCK_SIZE, device=DEVICE))
    print()

# Loss Curve 시각화
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(train_losses) + 1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, len(val_losses) + 1), val_losses, label="Val Loss", marker='s')
plt.title("TinyGPT Loss Curve (Financial News)")
plt.xlabel("Epoch")
plt.ylabel("Cross Entropy Loss")
plt.legend()
plt.grid(True)
plt.show()